In [1]:
from __future__ import print_function,division
from builtins import range

In [2]:
import numpy as np 

In [3]:
SMALL_ENOUGH =1e-3
GAMMA=.9

In [4]:
class Grid:
    def __init__(self,rows,cols,start):
        self.rows=rows
        self.cols=cols
        self.i=start[0]
        self.j=start[1]
    
    def set(self,rewards,actions):
        self.rewards=rewards
        self.actions=actions
    
    def set_state(self,s):
        self.i=s[0]
        self.j=s[1]
    
    def current_state(self):
        return (self.i,self.j)

    def is_terminal(self,s):
        return s not in self.actions

    def get_next_state(self,s,a):
        i,j=s[0],s[1]
        if a in self.actions[i,j]:
            if a=='U':
                i -=1
            if a=='D':
                i +=1
            if a=='L':
                j -=1
            if a=='R':
                j +=1
        return i,j

    def move (self,action):
        if action in self.actions[(self.i,self.j)]:
            if action =='U':
                self.i -=1
            if action =='D':
                self.i +=1
            if action =='R':
                self.j +=1
            if action =='L':
                self.j -=1
            
        return self.reward.get((self.i,self.j),0)

    def undo_move (self,action):
        if action =='U':
            self.i +=1
        if action =='D':
            self.i -=1
        if action =='R':
            self.j -=1
        if action =='L':
            self.j +=1
        assert(self.current_state() in self.all_stats())

    def game_over (self):
        return (self.i,self.j) not in self.actions

    def all_states(self):
        return set(self.actions.keys()) | set (self.rewards.keys())

In [5]:
def print_values(V,g):
    for i in range (g.rows):
        print("--------------------------")
        for j in range(g.cols):
            v=V.get((i,j),0)
            if v >=0:
                print(" %.2f |"% v,end="")
            else:
                print("%.2f |" %v,end="")
        print("")
        
        
def print_policy(P,g):
    for i in range(g.rows):
        print("----------------------------")
        for j in range(g.cols):
            a=P.get((i,j),' ')
            print(" %s |" %a,end="")
        print("")
    


In [6]:
def standard_grid():
    g=Grid(3,4,(2,0))
    rewards={(0,3):1,(1,3):-1}

    actions={
        (0,0):('D','R'),
        (0,1):('L','R'),
        (0,2):('L','R','D'),
        (1,0):('U','D'),
        (1,2):('R','U','D'),
        (2,0):('U','R'),
        (2,1):('L','R'),
        (2,2):('L','R','U'),
        (2,3):('L','U')
    }
    
    
    g.set(rewards,actions)
    return g
    

In [7]:
def get_transition_probs_and_rewards(grid):
    transition_prob={}
    rewards={}
    
    for i in range(grid.rows):
        for j in range(grid.cols):
            s=(i,j)
            if not grid.is_terminal(s):
                 for a in grid.actions[s]:
                     s2=grid.get_next_state(s,a)
                     transition_prob[(s,a,s2)]=1
                     if s2 in grid.rewards:
                         rewards[(s,a,s2)]=grid.rewards[s2]
                         
    return transition_prob, rewards

In [10]:
def evaluate_deterministic_policy(grid,policy):
    V={}
    #ACTION_SPACE=['U','D','L','R']
    for s in grid.all_states():
        V[s]=0
        
    it=0
    while True:
        biggest_change=0
        for s in grid.all_states():
    
            if not grid.is_terminal(s):
                old_v=V[s]
                new_v=0
                for a in grid.actions[s]:
                    for s2 in grid.all_states():
                        action_prob=1 if policy.get(s)==a else 0
                        
                        r=rewards.get((s,a,s2),0)
                        new_v +=action_prob *transition_prob.get((s,a,s2),0) *(r+GAMMA*V[s2])
                        
                V[s]=new_v
                biggest_change = max(biggest_change,np.abs(old_v-V[s]))
                    
        it +=1
        if biggest_change < SMALL_ENOUGH:
            break
    return V
            
            

In [13]:
if __name__=='__main__':
    grid=standard_grid()
    #ACTION_SPACE=['U','D','L','R']
    transition_prob,rewards=get_transition_probs_and_rewards(grid)

    
    print("rewards:")
    print_values(grid.rewards,grid)
    
    policy={}
    for s in grid.actions.keys():
        policy[s]=np.random.choice(grid.actions[s])
        
    print("initial policy:")
    print_policy(policy,grid)
    
    while True:
        V=evaluate_deterministic_policy(grid,policy)
        
        is_policy_converged=True
        for s in grid.actions.keys():
            old_a=policy[s]
            new_a=None
            best_value=float('-inf')
            
            for a in grid.actions[s]:
                v=0
                for s2 in grid.all_states():
                    r=rewards.get((s,a,s2),0)
                    v +=transition_prob.get((s,a,s2),0)*(r+GAMMA*V[s2])
                if v>best_value:
                    best_value=v 
                    new_a=a
                    
                    
            policy[s]=new_a
            if new_a !=old_a:
                is_policy_converged=False
        if is_policy_converged:
            break
        
    print("Values:")
    print_values(V,grid)
    print("policy:")
    print_policy(policy,grid)


rewards:
--------------------------
 0.00 | 0.00 | 0.00 | 1.00 |
--------------------------
 0.00 | 0.00 | 0.00 |-1.00 |
--------------------------
 0.00 | 0.00 | 0.00 | 0.00 |
initial policy:
----------------------------
 D | L | R |   |
----------------------------
 D |   | D |   |
----------------------------
 R | L | L | L |
Values:
--------------------------
 0.81 | 0.90 | 1.00 | 0.00 |
--------------------------
 0.73 | 0.00 | 0.90 | 0.00 |
--------------------------
 0.66 | 0.73 | 0.81 | 0.73 |
policy:
----------------------------
 R | R | R |   |
----------------------------
 U |   | U |   |
----------------------------
 U | R | U | L |
